# 02. Data Cleaning and Validation

## Project: Brazilian E-commerce Analysis

### Objective
This notebook documents the cleaning and validation of the Brazilian E-commerce datasets. The process uses separate working copies of the original CSV files to preserve data integrity and ensure that all cleaning decisions are traceable and reproducible.

The data-quality assessment conducted in the previous notebook informs the cleaning decisions applied here.

### Import libraries

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

### 1. Create copies of your CSV files

In [8]:
from pathlib import Path
import shutil

# Locate the original CSV files
raw_folder = Path("../data/raw")

# Create the working folder
working_folder = Path("../data/working")
working_folder.mkdir(parents=True, exist_ok=True)

# Copy the original CSV files
for file in raw_folder.glob("*.csv"):
    shutil.copy2(file, working_folder / file.name)

# Confirm the copies
print("Number of original CSV files:", len(list(raw_folder.glob("*.csv"))))
print("Number of working CSV files:", len(list(working_folder.glob("*.csv"))))

print("\nFiles copied:")
for file in sorted(working_folder.glob("*.csv")):
    print(file.name)

Number of original CSV files: 9
Number of working CSV files: 9

Files copied:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [9]:
from pathlib import Path
import pandas as pd

working_folder = Path("../data/working")

cleaned_datasets = {
    "Customers": pd.read_csv(working_folder / "olist_customers_dataset.csv"),
    "Orders": pd.read_csv(working_folder / "olist_orders_dataset.csv"),
    "Order Items": pd.read_csv(working_folder / "olist_order_items_dataset.csv"),
    "Order Payments": pd.read_csv(working_folder / "olist_order_payments_dataset.csv"),
    "Order Reviews": pd.read_csv(working_folder / "olist_order_reviews_dataset.csv"),
    "Products": pd.read_csv(working_folder / "olist_products_dataset.csv"),
    "Sellers": pd.read_csv(working_folder / "olist_sellers_dataset.csv"),
    "Category Translation": pd.read_csv(working_folder / "product_category_name_translation.csv"),
    "Geolocation": pd.read_csv(working_folder / "olist_geolocation_dataset.csv")
}

In [3]:
for name, df in cleaned_datasets.items():
    print(f"{name}: {df.shape}")

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Order Payments: (103886, 5)
Order Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Category Translation: (71, 2)
Geolocation: (1000163, 5)


In [4]:
working_datasets = cleaned_datasets.copy()

print("Working datasets loaded:")
for name, df in working_datasets.items():
    print(f"{name}: {df.shape}")

Working datasets loaded:
Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Order Payments: (103886, 5)
Order Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Category Translation: (71, 2)
Geolocation: (1000163, 5)


## 2. Date Conversion

Date columns were converted from text to datetime format to support
chronological comparisons, delivery duration calculations and time-based
analysis.

Missing date values were preserved and were not imputed.

In [11]:
# Identify date columns in each dataset
date_columns = {
    "Orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "Order Items": [
        "shipping_limit_date"
    ],
    "Order Reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

# Convert date columns to datetime
for dataset_name, columns in date_columns.items():
    for column in columns:
        working_datasets[dataset_name][column] = pd.to_datetime(
            working_datasets[dataset_name][column],
            errors="coerce"
        )

print("Date conversion completed.")

Date conversion completed.


In [12]:
for dataset_name, columns in date_columns.items():
    print(f"\n{dataset_name}")

    for column in columns:
        df = working_datasets[dataset_name]

        print(
            f"{column}: {df[column].dtype} | "
            f"Missing values: {df[column].isna().sum()}"
        )


Orders
order_purchase_timestamp: datetime64[ns] | Missing values: 0
order_approved_at: datetime64[ns] | Missing values: 160
order_delivered_carrier_date: datetime64[ns] | Missing values: 1783
order_delivered_customer_date: datetime64[ns] | Missing values: 2965
order_estimated_delivery_date: datetime64[ns] | Missing values: 0

Order Items
shipping_limit_date: datetime64[ns] | Missing values: 0

Order Reviews
review_creation_date: datetime64[ns] | Missing values: 0
review_answer_timestamp: datetime64[ns] | Missing values: 0


## 3. ZIP Code Prefix Standardisation

ZIP code prefixes in the customer, seller and geolocation datasets were
standardised to five-character strings.

Leading zeros were restored where they had been lost during numeric
conversion. This ensures consistent formatting across the datasets.

In [13]:
# Identify ZIP prefix columns
zip_columns = {
    "Customers": "customer_zip_code_prefix",
    "Sellers": "seller_zip_code_prefix",
    "Geolocation": "geolocation_zip_code_prefix"
}

# Restore five-character ZIP prefixes
for dataset_name, column in zip_columns.items():

    working_datasets[dataset_name][column] = (
        working_datasets[dataset_name][column]
        .astype(str)
        .str.zfill(5)
    )

print("ZIP prefix standardisation completed.")

ZIP prefix standardisation completed.


In [14]:
for dataset_name, column in zip_columns.items():

    df = working_datasets[dataset_name]

    print(f"\n{dataset_name}")
    print("Data type:", df[column].dtype)
    print("Prefix lengths:")
    print(df[column].str.len().value_counts().sort_index())
    print("Sample values:")
    print(df[column].head().tolist())


Customers
Data type: object
Prefix lengths:
customer_zip_code_prefix
5    99441
Name: count, dtype: int64
Sample values:
['14409', '09790', '01151', '08775', '13056']

Sellers
Data type: object
Prefix lengths:
seller_zip_code_prefix
5    3095
Name: count, dtype: int64
Sample values:
['13023', '13844', '20031', '04195', '12914']

Geolocation
Data type: object
Prefix lengths:
geolocation_zip_code_prefix
5    738332
Name: count, dtype: int64
Sample values:
['01037', '01046', '01046', '01041', '01035']


### 4. Gelocation Data Cleaning

In [15]:
# Check geolocation records in the current working DataFrame
print("Current working dataset:")
print("Rows:", len(working_datasets["Geolocation"]))
print("Exact duplicates:", working_datasets["Geolocation"].duplicated().sum())


# Check the original raw CSV file
raw_geolocation = pd.read_csv(
    "../data/raw/olist_geolocation_dataset.csv"
)

print("\nOriginal raw CSV:")
print("Rows:", len(raw_geolocation))
print("Exact duplicates:", raw_geolocation.duplicated().sum())

Current working dataset:
Rows: 738332
Exact duplicates: 0

Original raw CSV:
Rows: 1000163
Exact duplicates: 261831


### : Refresh the working CSV files

In [16]:
from pathlib import Path
import shutil

raw_folder = Path("../data/raw")
working_folder = Path("../data/working")

# Restore working files from the original raw CSV files
for file in raw_folder.glob("*.csv"):
    shutil.copy2(file, working_folder / file.name)

print("Working CSV files restored from raw data.")

Working CSV files restored from raw data.


### : Reload the working datasets

In [17]:
working_datasets = {
    "Customers": pd.read_csv(working_folder / "olist_customers_dataset.csv"),
    "Orders": pd.read_csv(working_folder / "olist_orders_dataset.csv"),
    "Order Items": pd.read_csv(working_folder / "olist_order_items_dataset.csv"),
    "Order Payments": pd.read_csv(working_folder / "olist_order_payments_dataset.csv"),
    "Order Reviews": pd.read_csv(working_folder / "olist_order_reviews_dataset.csv"),
    "Products": pd.read_csv(working_folder / "olist_products_dataset.csv"),
    "Sellers": pd.read_csv(working_folder / "olist_sellers_dataset.csv"),
    "Category Translation": pd.read_csv(working_folder / "product_category_name_translation.csv"),
    "Geolocation": pd.read_csv(working_folder / "olist_geolocation_dataset.csv")
}

print("Working datasets reloaded.")
for name, df in working_datasets.items():
    print(f"{name}: {df.shape}")

Working datasets reloaded.
Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Order Payments: (103886, 5)
Order Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Category Translation: (71, 2)
Geolocation: (1000163, 5)


### : Verify geolocation

In [18]:
print("Geolocation rows:", len(working_datasets["Geolocation"]))
print(
    "Exact duplicate rows:",
    working_datasets["Geolocation"].duplicated().sum()
)

Geolocation rows: 1000163
Exact duplicate rows: 261831


The 261,831 duplicate records have already been removed from the working CSV before our current cleaning notebook completed that operation.
For a reproducible project, I want data/working to contain exact copies of the original raw CSV files. Then our new cleaning notebook will perform and document every transformation itself.

#### : Reapply the date and ZIP code standardisation

In [21]:
# Standardise date columns
date_columns = {
    "Orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "Order Items": ["shipping_limit_date"],
    "Order Reviews": ["review_creation_date", "review_answer_timestamp"]
}

for dataset_name, columns in date_columns.items():
    for column in columns:
        working_datasets[dataset_name][column] = pd.to_datetime(
            working_datasets[dataset_name][column],
            errors="coerce"
        )

# Standardise ZIP code prefixes to five characters
zip_columns = {
    "Customers": "customer_zip_code_prefix",
    "Sellers": "seller_zip_code_prefix",
    "Geolocation": "geolocation_zip_code_prefix"
}

for dataset_name, column in zip_columns.items():
    working_datasets[dataset_name][column] = (
        working_datasets[dataset_name][column]
        .astype(str)
        .str.zfill(5)
    )

print("Date and ZIP code standardisation completed.")

Date and ZIP code standardisation completed.


In [22]:
for dataset_name, columns in date_columns.items():
    for column in columns:
        print(
            dataset_name,
            column,
            working_datasets[dataset_name][column].dtype,
            "Missing:",
            working_datasets[dataset_name][column].isna().sum()
        )

for dataset_name, column in zip_columns.items():
    print(
        dataset_name,
        "ZIP code lengths:",
        working_datasets[dataset_name][column].str.len().value_counts().to_dict()
    )

Orders order_purchase_timestamp datetime64[ns] Missing: 0
Orders order_approved_at datetime64[ns] Missing: 160
Orders order_delivered_carrier_date datetime64[ns] Missing: 1783
Orders order_delivered_customer_date datetime64[ns] Missing: 2965
Orders order_estimated_delivery_date datetime64[ns] Missing: 0
Order Items shipping_limit_date datetime64[ns] Missing: 0
Order Reviews review_creation_date datetime64[ns] Missing: 0
Order Reviews review_answer_timestamp datetime64[ns] Missing: 0
Customers ZIP code lengths: {5: 99441}
Sellers ZIP code lengths: {5: 3095}
Geolocation ZIP code lengths: {5: 1000163}


### : Remove exact duplicate geolocation records

In [23]:
# Remove exact duplicate records from the geolocation dataset

geolocation_before = len(working_datasets["Geolocation"])

working_datasets["Geolocation"] = (
    working_datasets["Geolocation"]
    .drop_duplicates()
    .copy()
)

geolocation_after = len(working_datasets["Geolocation"])

duplicates_removed = geolocation_before - geolocation_after

print("Geolocation cleaning results")
print("Rows before cleaning:", geolocation_before)
print("Rows after cleaning:", geolocation_after)
print("Exact duplicates removed:", duplicates_removed)

Geolocation cleaning results
Rows before cleaning: 1000163
Rows after cleaning: 738332
Exact duplicates removed: 261831


### 5. Standardise product column names

In [24]:
# Standardise product column names

working_datasets["Products"].rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    },
    inplace=True
)

print("Updated product columns:")
print(working_datasets["Products"].columns.tolist())

Updated product columns:
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


### 6. Validate the cleaned datasets

In [25]:
# Validate cleaned dataset dimensions and duplicate records

for name, df in working_datasets.items():
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Exact duplicate rows:", df.duplicated().sum())


Customers
Rows: 99441
Columns: 5
Exact duplicate rows: 0

Orders
Rows: 99441
Columns: 8
Exact duplicate rows: 0

Order Items
Rows: 112650
Columns: 7
Exact duplicate rows: 0

Order Payments
Rows: 103886
Columns: 5
Exact duplicate rows: 0

Order Reviews
Rows: 99224
Columns: 7
Exact duplicate rows: 0

Products
Rows: 32951
Columns: 9
Exact duplicate rows: 0

Sellers
Rows: 3095
Columns: 4
Exact duplicate rows: 0

Category Translation
Rows: 71
Columns: 2
Exact duplicate rows: 0

Geolocation
Rows: 738332
Columns: 5
Exact duplicate rows: 0


### 7. Export the cleaned datasets

In [26]:
# Create the cleaned data folder

cleaned_folder = Path("../data/cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True)

print("Cleaned data folder is ready:", cleaned_folder.resolve())

Cleaned data folder is ready: C:\Users\hp\Data Science and AI\NexaScale Applied AI fellowship\Brazilian_Ecommerce_Analysis\data\cleaned


In [27]:
# Export cleaned datasets as CSV files

file_names = {
    "Customers": "olist_customers_dataset.csv",
    "Orders": "olist_orders_dataset.csv",
    "Order Items": "olist_order_items_dataset.csv",
    "Order Payments": "olist_order_payments_dataset.csv",
    "Order Reviews": "olist_order_reviews_dataset.csv",
    "Products": "olist_products_dataset.csv",
    "Sellers": "olist_sellers_dataset.csv",
    "Category Translation": "product_category_name_translation.csv",
    "Geolocation": "olist_geolocation_dataset.csv"
}

for dataset_name, file_name in file_names.items():
    output_path = cleaned_folder / file_name

    working_datasets[dataset_name].to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {file_name} | Rows: {len(working_datasets[dataset_name])}")

print("\nAll cleaned datasets exported successfully.")

Saved: olist_customers_dataset.csv | Rows: 99441
Saved: olist_orders_dataset.csv | Rows: 99441
Saved: olist_order_items_dataset.csv | Rows: 112650
Saved: olist_order_payments_dataset.csv | Rows: 103886
Saved: olist_order_reviews_dataset.csv | Rows: 99224
Saved: olist_products_dataset.csv | Rows: 32951
Saved: olist_sellers_dataset.csv | Rows: 3095
Saved: product_category_name_translation.csv | Rows: 71
Saved: olist_geolocation_dataset.csv | Rows: 738332

All cleaned datasets exported successfully.


### Verify exported CSV files

In [28]:
# Verify exported cleaned datasets

for dataset_name, file_name in file_names.items():
    exported_df = pd.read_csv(cleaned_folder / file_name)

    print(
        f"{dataset_name}: "
        f"{len(exported_df)} rows | "
        f"{len(exported_df.columns)} columns"
    )

Customers: 99441 rows | 5 columns
Orders: 99441 rows | 8 columns
Order Items: 112650 rows | 7 columns
Order Payments: 103886 rows | 5 columns
Order Reviews: 99224 rows | 7 columns
Products: 32951 rows | 9 columns
Sellers: 3095 rows | 4 columns
Category Translation: 71 rows | 2 columns
Geolocation: 738332 rows | 5 columns
